In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import mr4mp
import itertools
from random import choice
from string import ascii_lowercase
import math
import multiprocessing
multiprocessing.set_start_method("fork", force=True)

In [ ]:
with open("web-Stanford.txt") as f:
    lines = f.readlines()

In [14]:
graph = nx.DiGraph()

for line in lines:
    if line.startswith('#'):
        continue
    edge = line.split()
    node1 = int(edge[0])
    node2 = int(edge[1])
    graph.add_edge(node1, node2)


print("Number of nodes:", graph.number_of_nodes())
print("Number of edges:", graph.number_of_edges())

nodes = sorted(list(graph.nodes()))
edges = sorted(list(graph.edges()))

Number of nodes: 281903
Number of edges: 2312497


In [15]:
nodes = nodes[:200]
nodes_set = set(nodes) 

out_degree = {}
for node in nodes:
    degree = graph.out_degree(node)
    if degree > 0:
        out_degree[node] = 1 / degree
    else:
        out_degree[node] = 0


node_indices = {node: i for i, node in enumerate(nodes)}


rni = {i: v for v, i in node_indices.items()} #Reverse node index per recuperare l'ID originale


edges = [edge for edge in sorted(list(graph.edges())) if edge[0] in nodes_set and edge[1] in nodes_set]

print(f"Subset creato: {len(nodes)} nodi e {len(edges)} archi.")

Subset creato: 200 nodi e 2 archi.


In [16]:
n = len(nodes)
matrix = np.zeros((n, n))

for edge in edges:
    value = out_degree[edge[0]]
    row = node_indices[edge[1]]
    column = node_indices[edge[0]]
    matrix[row][column] = value


final_matrix = (matrix * 0.85) + (1 - 0.85) / len(matrix)

In [17]:
def transition(matrix):
    li = []
    for i, line in enumerate(matrix):
        for j, element in enumerate(line):
            li.append([(i, j), element])
    return li

c = transition(final_matrix)

for element in c:
    element.append((element[0][0], 1 / len(matrix)))

def update_matrix(matrix, vector):
    for line in matrix:
        for k in vector:
            if line[0][0] == k[0]:
                line[1] = k[1]

In [26]:
def update_matrix(matrix, vector):###faster matrix update

    vec_dict = {k[0]: k for k in vector}
    
    for line in matrix:
        nodo_id = line[0][0]
        if nodo_id in vec_dict:
            line[1] = vec_dict[nodo_id]

In [19]:
print(f"Tipo di c: {type(c)}")
print(f"Lunghezza di c: {len(c)}")
print(f"Esempio del primo elemento (c[0]): {c[0]}")
print(f"Tipo del primo elemento: {type(c[0])}")

Tipo di c: <class 'list'>
Lunghezza di c: 40000
Esempio del primo elemento (c[0]): [(0, 0), 0.0007500000000000001, (0, 0.005)]
Tipo del primo elemento: <class 'list'>


In [27]:
c = []
n = len(nodes)
initial_pr = 1 / n 

for i, line in enumerate(final_matrix):
    for j, element_val in enumerate(line):
        c.append([ (i, j, element_val), (i, initial_pr) ])

In [ ]:
def mapp(block):
    if block[0][0] == block[1][0]:
        t = {(block[0][0], block[0][1]): block[0][2] * block[1][1]}
    
    return {(block[0][0], block[0][1]): block[0][2] * block[1][1]}

def reducer(pairs1, pairs2):
    #print("pairs1 keys:", list(pairs1.keys()))
    #print("pairs2 keys:", list(pairs2.keys()))
    
    # Unione delle chiavi dei due dizionari
    keys = pairs1.keys() | pairs2.keys()
    
    #print('pair1', pairs1)
    #print('pair2', pairs2)
    
    merged = {k: (((pairs1.get(k, 0) + pairs2.get(k, 0)))) for k in keys}
    
    #print(keys)
    #print(merged)
    return merged

In [24]:
def update_matrix(matrix, vector):
    # Trasformiamo il vettore in un dizionario per trovare i dati all'istante
    # k[0] è l'indice del nodo, k è la tupla (indice, valore)
    vec_dict = {k[0]: k for k in vector}
    
    for line in matrix:
        # Invece di un altro ciclo for, cerchiamo direttamente il nodo
        nodo_id = line[0][0]
        if nodo_id in vec_dict:
            line[1] = vec_dict[nodo_id]

In [25]:
from timeit import default_timer

start = default_timer()
counter = 0

while counter < 5:
    pool = mr4mp.pool()
    out = pool.mapreduce(mapp, reducer, c)
    pool.close()
    
    print(f"Iteration number: {counter} finished in {default_timer() - start}s using {len(pool)} process(es).")
    
    result = {}
    for key, value in out.items():
        if key not in result:
            result[key] = value
        else:
            result[key] += value
            
    # Normalizzazione e aggiornamento
    sum_pr = sum(result.values())
    output = [(x, y / sum_pr) for x, y in result.items()]
    update_matrix(c, output)
    
    counter += 1

print("this is the final pangeRank: " + str(output))

Iteration number: 0 finished in 75.66482257500002s using 4 process(es).
Iteration number: 1 finished in 144.33354327099983s using 4 process(es).
Iteration number: 2 finished in 212.25088492999998s using 4 process(es).
Iteration number: 3 finished in 278.22061706299996s using 4 process(es).
Iteration number: 4 finished in 344.550691466s using 4 process(es).
this is the final pangeRank: [((90, 42), 2.4972118769220103e-05), ((150, 93), 2.4972118769220103e-05), ((53, 160), 2.4972118769220103e-05), ((92, 88), 2.4972118769220103e-05), ((143, 53), 2.4972118769220103e-05), ((44, 47), 2.4972118769220103e-05), ((85, 48), 2.4972118769220103e-05), ((104, 98), 2.4972118769220103e-05), ((145, 99), 2.4972118769220103e-05), ((7, 165), 2.4972118769220103e-05), ((48, 166), 2.4972118769220103e-05), ((97, 58), 2.4972118769220103e-05), ((157, 109), 2.4972118769220103e-05), ((196, 37), 2.4972118769220103e-05), ((60, 176), 2.4972118769220103e-05), ((99, 104), 2.4972118769220103e-05), ((159, 155), 2.497211876